In [1]:
import os
from PIL import Image

base_dir = "dataset/WiderPerson"

images_src = os.path.join(base_dir, "Images")
annotations_src = os.path.join(base_dir, "Annotations")

images_train = os.path.join(base_dir, "images/train")
images_val = os.path.join(base_dir, "images/val")

labels_train = os.path.join(base_dir, "labels/train")
labels_val = os.path.join(base_dir, "labels/val")

for folder in [images_train, images_val, labels_train, labels_val]:
    os.makedirs(folder, exist_ok=True)

In [2]:
with open(os.path.join(base_dir, "train.txt")) as f:
    train_ids = [line.strip() for line in f]

with open(os.path.join(base_dir, "val.txt")) as f:
    val_ids = [line.strip() for line in f]

print("Train images:", len(train_ids))
print("Validation images:", len(val_ids))

Train images: 8000
Validation images: 1000


In [3]:
import shutil

def convert_annotation(image_id, split):

    img_path = os.path.join(images_src, image_id + ".jpg")
    ann_path = os.path.join(annotations_src, image_id + ".jpg.txt")

    img = Image.open(img_path)
    w, h = img.size

    yolo_lines = []

    with open(ann_path) as f:
        lines = f.readlines()

    # Skip first line (object count)
    for line in lines[1:]:

        parts = line.strip().split()

        cls = int(parts[0])

        xmin = float(parts[1])
        ymin = float(parts[2])
        xmax = float(parts[3])
        ymax = float(parts[4])

        # Single-class pedestrian detector
        cls_id = 0

        x_center = ((xmin + xmax) / 2) / w
        y_center = ((ymin + ymax) / 2) / h

        box_w = (xmax - xmin) / w
        box_h = (ymax - ymin) / h

        yolo_lines.append(
            f"{cls_id} {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f}"
        )

    if split == "train":
        img_dst = images_train
        label_dst = labels_train
    else:
        img_dst = images_val
        label_dst = labels_val

    shutil.copy(img_path, img_dst)

    with open(os.path.join(label_dst, image_id + ".txt"), "w") as f:
        f.write("\n".join(yolo_lines))

In [4]:
for image_id in train_ids:

    img_path = os.path.join(images_src, image_id + ".jpg")
    ann_path = os.path.join(annotations_src, image_id + ".jpg.txt")

    if not os.path.exists(img_path):
        continue

    if not os.path.exists(ann_path):
        continue

    convert_annotation(image_id, "train")

for image_id in val_ids:

    img_path = os.path.join(images_src, image_id + ".jpg")
    ann_path = os.path.join(annotations_src, image_id + ".jpg.txt")

    if not os.path.exists(img_path):
        continue

    if not os.path.exists(ann_path):
        continue

    convert_annotation(image_id, "val")

print("Conversion completed")

Conversion completed


In [5]:
!python yolov5/train.py --help

usage: train.py [-h] [--weights WEIGHTS] [--cfg CFG] [--data DATA] [--hyp HYP]
                [--epochs EPOCHS] [--batch-size BATCH_SIZE] [--imgsz IMGSZ]
                [--rect] [--resume [RESUME]] [--nosave] [--noval]
                [--noautoanchor] [--noplots] [--evolve [EVOLVE]]
                [--evolve_population EVOLVE_POPULATION]
                [--resume_evolve RESUME_EVOLVE] [--bucket BUCKET]
                [--cache [CACHE]] [--image-weights] [--device DEVICE]
                [--multi-scale] [--single-cls] [--optimizer {SGD,Adam,AdamW}]
                [--sync-bn] [--workers WORKERS] [--project PROJECT]
                [--name NAME] [--exist-ok] [--quad] [--cos-lr]
                [--label-smoothing LABEL_SMOOTHING] [--patience PATIENCE]
                [--freeze FREEZE [FREEZE ...]] [--save-period SAVE_PERIOD]
                [--seed SEED] [--local_rank LOCAL_RANK] [--entity ENTITY]
                [--upload_dataset [UPLOAD_DATASET]]
                [--bbox_interval BBOX_

In [6]:
import os

for f in [
    "dataset/WiderPerson/images/train.cache",
    "dataset/WiderPerson/images/val.cache"
]:
    if os.path.exists(f):
        os.remove(f)

In [8]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [9]:
!python yolov5/detect.py \
--weights yolov5/runs/train/exp13/weights/best.pt \
--source dataset/WiderPerson/images/val

detect: weights=['yolov5/runs/train/exp13/weights/best.pt'], source=dataset/WiderPerson/images/val, data=yolov5\data\coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=yolov5\runs\detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5  v7.0-484-g70b964b6 Python-3.13.9 torch-2.12.0+cpu CPU

Fusing layers... 
Model summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
image 1/1000 C:\Users\vavil\project\dataset\WiderPerson\images\val\000041.jpg: 416x640 55 persons, 434.8ms
image 2/1000 C:\Users\vavil\project\dataset\WiderPerson\images\val\000044.jpg: 480x640 20 persons, 453.5ms
image 3/1000 C:\Users\vavil\project\dataset\WiderPerson\images\val\000045.jpg: 288x640 13 per

In [10]:
!python yolov5/val.py \
--weights yolov5/runs/train/exp13/weights/best.pt \
--data dataset/WiderPerson/widerperson.yaml \
--img 640

val: data=dataset/WiderPerson/widerperson.yaml, weights=['yolov5/runs/train/exp13/weights/best.pt'], batch_size=32, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=val, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=yolov5\runs\val, name=exp, exist_ok=False, half=False, dnn=False
YOLOv5  v7.0-484-g70b964b6 Python-3.13.9 torch-2.12.0+cpu CPU

Fusing layers... 
Model summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs

val: Scanning C:\Users\vavil\project\dataset\WiderPerson\labels\val...:   0%|          | 0/1000 [00:00<?, ?it/s]
val: Scanning C:\Users\vavil\project\dataset\WiderPerson\labels\val... 1 images, 0 backgrounds, 0 corrupt:   0%|          | 1/1000 [00:32<8:53:26, 32.04s/it]
val: Scanning C:\Users\vavil\project\dataset\WiderPerson\labels\val... 24 images, 0 backgrounds, 0 corrupt:   2%|2         | 24/1000 [00:32<15:28,  1.05it/s]
val: Scanning C:\Users\v

In [ ]:
!python yolov5/train.py --img 640 --batch 1 --epochs 10 --workers 0 --data dataset/WiderPerson/widerperson.yaml --weights yolov5s.pt

In [10]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

!set KMP_DUPLICATE_LIB_OK=TRUE && python yolov5/detect.py --weights yolov5/runs/train/exp15/weights/best.pt --source dataset/WiderPerson/images/val

detect: weights=['yolov5/runs/train/exp15/weights/best.pt'], source=dataset/WiderPerson/images/val, data=yolov5\data\coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=yolov5\runs\detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5  v7.0-484-g70b964b6 Python-3.13.9 torch-2.12.0+cpu CPU

Fusing layers... 
Model summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
image 1/1000 C:\Users\vavil\project\dataset\WiderPerson\images\val\000041.jpg: 416x640 59 persons, 420.7ms
image 2/1000 C:\Users\vavil\project\dataset\WiderPerson\images\val\000044.jpg: 480x640 20 persons, 421.7ms
image 3/1000 C:\Users\vavil\project\dataset\WiderPerson\images\val\000045.jpg: 288x640 13 per

In [11]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

!python yolov5/val.py --weights yolov5/runs/train/exp15/weights/best.pt --data dataset/WiderPerson/widerperson.yaml --img 640

val: data=dataset/WiderPerson/widerperson.yaml, weights=['yolov5/runs/train/exp15/weights/best.pt'], batch_size=32, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=val, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=yolov5\runs\val, name=exp, exist_ok=False, half=False, dnn=False
YOLOv5  v7.0-484-g70b964b6 Python-3.13.9 torch-2.12.0+cpu CPU

Fusing layers... 
Model summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs

val: Scanning C:\Users\vavil\project\dataset\WiderPerson\labels\val...:   0%|          | 0/1000 [00:00<?, ?it/s]
val: Scanning C:\Users\vavil\project\dataset\WiderPerson\labels\val... 1 images, 0 backgrounds, 0 corrupt:   0%|          | 1/1000 [00:30<8:32:48, 30.80s/it]
val: Scanning C:\Users\vavil\project\dataset\WiderPerson\labels\val... 6 images, 0 backgrounds, 0 corrupt:   1%|          | 6/1000 [00:30<1:02:59,  3.80s/it]
val: Scanning C:\Users\v

In [ ]:
!python -m ultralytics detect train data=dataset/WiderPerson/widerperson.yaml model=yolov8n.pt epochs=10 imgsz=640 batch=1 workers=0

In [ ]:
!yolo detect predict model=runs/detect/train/weights/best.pt source=dataset/WiderPerson/images/val

In [ ]:
!yolo detect train data=dataset/WiderPerson/widerperson.yaml model=yolov10n.pt epochs=3 imgsz=640 batch=1 workers=0

In [ ]:
!yolo detect predict model=runs/detect/train4/weights/best.pt source=dataset/WiderPerson/images/val

In [ ]:
!yolo detect val model=runs/detect/train4/weights/best.pt data=dataset/WiderPerson/widerperson.yaml